In [1]:
!git clone https://github.com/beltrewilton/mspconv_ftlab.git
# !git pull https://github.com/beltrewilton/mspconv_ftlab.git

fatal: destination path 'mspconv_ftlab' already exists and is not an empty directory.


In [2]:
%cd /content/mspconv_ftlab/
!mkdir cache

/content/mspconv_ftlab
mkdir: cannot create directory ‘cache’: File exists


In [3]:
!pip install -r requirements.txt

  Cloning https://github.com/beltrewilton/VAD.git to /tmp/pip-req-build-g4_mb2xl
  Running command git clone --filter=blob:none --quiet https://github.com/beltrewilton/VAD.git /tmp/pip-req-build-g4_mb2xl
  Resolved https://github.com/beltrewilton/VAD.git to commit 3dbf5a9718add6af0f3162725becf93cf50ca941
  Preparing metadata (setup.py) ... done
  Using cached lightning-2.2.1-py3-none-any.whl (2.1 MB)
  Using cached torchmetrics-1.3.2-py3-none-any.whl (841 kB)
  Using cached flashlight_text-0.0.7-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.3 MB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.9 MB/s eta 0:00:00
  

In [4]:
!gdown  1rvZTYm_ltO13WYGCeamwbQFQOQBjJfgG -O data/class_input_features_development_fixed.pkl
!gdown  1QjoHfzMtQeqLl0bXE6MH400r7gMadX1y -O data/class_input_features_test_fixed.pkl
!gdown  1Z6IFfR2K77ranfTmT4JElsDYQRo8LAug -O data/class_input_features_train_fixed.pkl


!gdown 1PSZdbOmvLw92dSVmM93-c6VKpKrCXdMb -O audiosegments.tar.gz

!tar -xzf audiosegments.tar.gz

Downloading...
From: https://drive.google.com/uc?id=1rvZTYm_ltO13WYGCeamwbQFQOQBjJfgG
To: /content/mspconv_ftlab/data/class_input_features_development_fixed.pkl
100% 134k/134k [00:00<00:00, 75.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QjoHfzMtQeqLl0bXE6MH400r7gMadX1y
To: /content/mspconv_ftlab/data/class_input_features_test_fixed.pkl
100% 187k/187k [00:00<00:00, 78.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Z6IFfR2K77ranfTmT4JElsDYQRo8LAug
To: /content/mspconv_ftlab/data/class_input_features_train_fixed.pkl
100% 464k/464k [00:00<00:00, 103MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1PSZdbOmvLw92dSVmM93-c6VKpKrCXdMb
From (redirected): https://drive.google.com/uc?id=1PSZdbOmvLw92dSVmM93-c6VKpKrCXdMb&confirm=t&uuid=3e9f9b4a-4683-4c5a-bf5a-02e6ebb7fc27
To: /content/mspconv_ftlab/audiosegments.tar.gz
100% 2.28G/2.28G [00:15<00:00, 143MB/s]


In [5]:
# !gdown --id 1-tapzQbHQVZ8tPX-u23izY7K1f8IxUWj -O last.ckpt


In [6]:
!ls -ltrh

total 2.2G
drwxr-xr-x 2  501 staff 1.3M Apr  5 04:41 audiosegments
-rw-r--r-- 1 root root  1.1K Apr  7 05:47 README.md
drwxr-xr-x 2 root root  4.0K Apr  7 05:47 models
-rw-r--r-- 1 root root    99 Apr  7 05:47 requirements.txt
drwxr-xr-x 2 root root  4.0K Apr  7 05:47 notebooks
-rw-r--r-- 1 root root  1.5K Apr  7 05:47 ubicacion_features_transcris.ipynb
-rw-r--r-- 1 root root  1.8K Apr  7 05:47 todo.txt
drwxr-xr-x 2 root root  4.0K Apr  7 05:47 src
drwxr-xr-x 2 root root  4.0K Apr  7 05:47 cache
drwxr-xr-x 3 root root  4.0K Apr  7 05:53 data
-rw-r--r-- 1 root root  2.2G Apr  7 05:53 audiosegments.tar.gz


In [7]:
!cd logs; rm -rf lightning_logs

/bin/bash: line 1: cd: logs: No such file or directory


In [8]:
# from google.colab import drive
# drive.mount('/content/gdrive')

# # !mkdir /content/gdrive/MyDrive/output/
# # !mkdir /content/gdrive/MyDrive/output/logs
# !ls -ltrh /content/gdrive/MyDrive/output/


In [9]:
# %load_ext tensorboard
# %tensorboard --logdir /content/gdrive/MyDrive/output/logs/

In [ ]:
import sys
import os
import torch
from torch.utils.data import DataLoader
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from models.data_processor import MSPDataProcessor, MSP_PATH, ROOT, AUDIO_SEGMENTS
from models.dataset_utils import MSPDataset
from models.architecture import Wav2vec2ModelWrapper, Wav2vec2ModelWrapperForClassification, MSPImplementation, MSPImplementationForClassification, Timem

os.environ['HF_HOME'] = f'{ROOT}/cache'
os.environ['HF_DATASETS_CACHE'] = f'{ROOT}/cache'
LOG_DIR = f"{ROOT}/logs"
# LOG_DIR = "/content/gdrive/MyDrive/output/logs"


batch_size = 8
chunk_size = 5
overlap = 0.5
num_workers = 7
checkpoint_name = "facebook/wav2vec2-xls-r-300m"
train_mode = True
lr = 1e-5
epochs = 30 #10


def get_loaders(batch_size: int, chunk_size: int, overlap: float, num_workers: int = 0 ):
    datapros_train = MSPDataProcessor(msp_path=MSP_PATH, chunk_size=chunk_size, overlap=overlap, split="Train", verbose=True)
    datapros_test = MSPDataProcessor(msp_path=MSP_PATH, chunk_size=chunk_size, overlap=overlap, split="Test", verbose=True)
    datapros_dev = MSPDataProcessor(msp_path=MSP_PATH, chunk_size=chunk_size, overlap=overlap, split="Development", verbose=True)

    dataset_train = MSPDataset(input_features=datapros_train.load_input_features())
    dataset_test = MSPDataset(input_features=datapros_test.load_input_features())
    dataset_dev = MSPDataset(input_features=datapros_dev.load_input_features())

    loader_train = DataLoader(
        dataset=dataset_train,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        persistent_workers=True,
        collate_fn=dataset_train.seqCollate,
    )

    loader_test = DataLoader(
        dataset=dataset_test,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=dataset_test.seqCollate,
    )

    loader_dev = DataLoader(
        dataset=dataset_dev,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=dataset_dev.seqCollate,
    )

    return loader_train, loader_dev, loader_test


class MeasureCallback(L.Callback):
    def __init__(self):
        self.time_batch = Timem()
        self.time_back = Timem()
        self.time_epoch = Timem()
        self.time_train = Timem()

    def on_train_start(self, trainer, pl_module):
        self.time_train.start("***** on_train_start *****")

    def on_train_end(self, trainer, pl_module):
        self.time_train.end("***** on_train_end *****")

    def on_train_batch_start(
        self, trainer, pl_module,  batch, batch_idx
    ):
        self.time_batch.start("")
        # torch.cuda.empty_cache()
        # import gc
        # gc.collect()

    def on_train_batch_end(
        self, trainer, pl_module, outputs, batch, batch_idx
    ):
        execution_time = self.time_batch.end(f"on_train_batch_end batch_idx:{batch_idx}")
        pl_module.log("train_mins_per_batch", execution_time / 60)

    def on_before_backward(self, trainer, pl_module, loss):
        self.time_back.start("")

    def on_after_backward(self, trainer, pl_module):
        self.time_back.end("on_after_backward")

    def on_train_epoch_start(self, trainer, pl_module):
        self.time_epoch.start("")

    def on_train_epoch_end(self, trainer, pl_module):
        execution_time = self.time_epoch.end("on_train_epoch_end")
        pl_module.log("train_mins_per_epoch", execution_time / 60)

    # def on_before_optimizer_step(
    #     self, trainer, pl_module, optimizer: torch.optim.AdamW
    # ) :
    #     # Now let's inspect the optimizer's state
    #     for group in optimizer.param_groups:
    #         for param in group['params']:
    #             print(f"param.shape:{param.shape} param.grad:{param.grad}")


if __name__ == "__main__":
    loader_train, loader_dev, loader_test = get_loaders(batch_size, chunk_size, overlap, num_workers)

    is_classification = True
    if is_classification:
        model = Wav2vec2ModelWrapperForClassification(checkpoint_name=checkpoint_name, train_mode=train_mode)
        msp_impl_model = MSPImplementationForClassification(model=model, lr=lr, train_mode=train_mode)
    else:
        pass

    checkpoint_callback = ModelCheckpoint(save_top_k=1, mode="min", monitor="train_loss", save_last=True)
    time_measure_callback = MeasureCallback()

  #trainer = pl.Trainer(max_epochs=10, resume_from_checkpoint='./checkpoints/blahblah.ckpt')

    trainer = L.Trainer(
        max_epochs=epochs,
        check_val_every_n_epoch=1,
        accelerator="auto",
        devices="auto",
        default_root_dir=LOG_DIR,
        callbacks=[checkpoint_callback, time_measure_callback],
        log_every_n_steps=1,
        num_sanity_val_steps=0,
    )

    trainer.fit(
        model=msp_impl_model,
        train_dataloaders=loader_train,
        val_dataloaders=loader_test,
        # ckpt_path="/content/mspconv_ftlab/last.ckpt",
    )

    train_acc = trainer.test(model=msp_impl_model, dataloaders=loader_train, ckpt_path="best")[0]["test_acc"]
    val_acc = trainer.test(model=msp_impl_model, dataloaders=loader_dev, ckpt_path="best")[0]["test_acc"]
    test_acc = trainer.test(model=msp_impl_model, dataloaders=loader_test, ckpt_path="best")[0]["test_acc"]

    print(
        f"\nTrain Acc {train_acc*100:.2f}%  "
        f"Val Acc {val_acc*100:.2f}%.     "
        f"Test Acc {test_acc*100:.2f}%.  "
    )






####################################################################################################
Inputs pre-loaded: /content/mspconv_ftlab/data/class_input_features_train_fixed.pkl
####################################################################################################

inputs:4024, labels:4024

####################################################################################################
Inputs pre-loaded: /content/mspconv_ftlab/data/class_input_features_test_fixed.pkl
####################################################################################################

inputs:1640, labels:1640

####################################################################################################
Inputs pre-loaded: /content/mspconv_ftlab/data/class_input_features_development_fixed.pkl
####################################################################################################

inputs:1096, labels:1096


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 7 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/wav2vec2-xls-r-300m were not used when initializing Wav2Vec2ForPreTraining: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForPreTraining from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForPreTraining from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForPreTraining were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-300m and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You

...

Time ended :on_train_epoch_end, execution time: 9.258774665991465 minutes
y_hat      : tensor([16, 21, 23, 20, 20, 14, 21, 16], device='cuda:0')
true_labels: tensor([16, 19, 23, 20, 10,  9, 21, 16], device='cuda:0')


y_hat      : tensor([21, 14, 21, 19, 23, 19, 11, 20], device='cuda:0')
true_labels: tensor([21, 21, 21, 19, 20, 19, 11, 20], device='cuda:0')


y_hat      : tensor([14, 20, 21, 21, 20, 20, 10, 23], device='cuda:0')
true_labels: tensor([14, 20, 21, 21, 20, 20, 10, 20], device='cuda:0')


y_hat      : tensor([21, 14, 21, 16, 20, 14, 20, 10], device='cuda:0')
true_labels: tensor([21, 14, 21, 16, 20, 14, 20, 10], device='cuda:0')


y_hat      : tensor([23, 20, 20, 19, 19, 16, 14, 20], device='cuda:0')
true_labels: tensor([12, 20, 20, 19, 19, 16,  9, 20], device='cuda:0')


y_hat      : tensor([23, 16, 16, 21, 14, 20, 23, 16], device='cuda:0')
true_labels: tensor([23, 16, 16, 21, 14, 20, 23, 16], device='cuda:0')


y_hat      : tensor([19, 19, 14, 19, 21, 21, 14, 20], device='

Validation: |          | 0/? [00:00<?, ?it/s]

Time ended :on_train_epoch_end, execution time: 9.26778914531072 minutes
y_hat      : tensor([19, 21, 14, 16, 19, 14, 23, 11], device='cuda:0')
true_labels: tensor([19, 21, 14, 16, 19, 14, 23,  9], device='cuda:0')


y_hat      : tensor([23, 21, 19, 21, 14, 23, 14,  1], device='cuda:0')
true_labels: tensor([23, 21, 19, 21, 14, 23, 14,  1], device='cuda:0')


y_hat      : tensor([23, 20, 14, 19, 20, 14, 20, 20], device='cuda:0')
true_labels: tensor([23, 20, 14, 19, 20, 14, 20, 20], device='cuda:0')


y_hat      : tensor([19, 21, 14, 20, 20, 14, 15, 21], device='cuda:0')
true_labels: tensor([19, 21, 14, 21, 20, 14, 15, 21], device='cuda:0')


y_hat      : tensor([21, 19,  2, 14, 19, 14, 23, 14], device='cuda:0')
true_labels: tensor([20, 19,  2, 14, 19, 14, 23, 14], device='cuda:0')


y_hat      : tensor([16, 14, 14, 19, 20, 14, 14, 19], device='cuda:0')
true_labels: tensor([14, 14, 14, 10, 20, 14, 14, 19], device='cuda:0')


y_hat      : tensor([20, 14, 21, 14, 16, 20, 14, 20], device='c

Validation: |          | 0/? [00:00<?, ?it/s]

Time ended :on_train_epoch_end, execution time: 9.26874644756317 minutes
y_hat      : tensor([21, 20, 20, 14, 14,  1, 21, 20], device='cuda:0')
true_labels: tensor([21, 20, 20, 14, 14,  1, 21, 20], device='cuda:0')




In [ ]:

!rsync -ah --progress logs/lightning_logs/version_0/checkpoints/last.ckpt /content/gdrive/MyDrive/output/
